###Fase 3.1

## 📦 Importações

- **pandas** → manipulação de dados
- **sqlite3** → criação e consulta do banco de dados (nativo do Python)
- **sklearn.preprocessing.StandardScaler** → normalização para o K-Means
- **os** → gerenciamento de arquivos
"""

In [ ]:
import pandas as pd
import sqlite3
import os
from sklearn.preprocessing import StandardScaler

print("✅ Bibliotecas importadas.")

## 📂 Carregamento do Dataset Já Tratado

Carregamos o `talabat_tratado.csv` — resultado do ETL feito anteriormente,
já sem os atributos desnecessários e com o atributo `Coupon_Used` criado.

Este arquivo possui **100.000 registros** e **18 colunas**.

In [ ]:
CAMINHO_CSV = "talabat_tratado.csv"

df = pd.read_csv(CAMINHO_CSV)

display(df.head())
print(f"\nDimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

## 🗄️ INGESTÃO — Criação do Banco de Dados SQLite

Criamos o banco `logisfast.db` com as **4 tabelas** do modelo relacional:
`CLIENTES`, `RESTAURANTES`, `ENTREGADORES` e `PEDIDOS` (tabela fato).

As dimensões são inseridas antes da tabela fato para garantir a
integridade referencial das chaves estrangeiras.

In [ ]:
NOME_BANCO = "logisfast.db"

if os.path.exists(NOME_BANCO):
    os.remove(NOME_BANCO)

conn   = sqlite3.connect(NOME_BANCO)
cursor = conn.cursor()

cursor.executescript("""
    PRAGMA foreign_keys = ON;

    CREATE TABLE IF NOT EXISTS CLIENTES (
        User_ID         TEXT PRIMARY KEY,
        City            TEXT,
        Payment_Method  TEXT
    );

    CREATE TABLE IF NOT EXISTS RESTAURANTES (
        Restaurant_ID   INTEGER PRIMARY KEY,
        City            TEXT,
        Restaurant_Lat  REAL,
        Restaurant_Lon  REAL
    );

    CREATE TABLE IF NOT EXISTS ENTREGADORES (
        Driver_ID            INTEGER PRIMARY KEY,
        Driver_Vehicle       TEXT,
        Driver_Availability  TEXT
    );

    CREATE TABLE IF NOT EXISTS PEDIDOS (
        Order_ID                   INTEGER PRIMARY KEY,
        User_ID                    TEXT    REFERENCES CLIENTES(User_ID),
        Restaurant_ID              INTEGER REFERENCES RESTAURANTES(Restaurant_ID),
        Driver_ID                  INTEGER REFERENCES ENTREGADORES(Driver_ID),
        Item_Name                  TEXT,
        Quantity                   INTEGER,
        Total_Price                REAL,
        Order_Time                 TEXT,
        Delivery_Duration_Minutes  INTEGER,
        Order_Status               TEXT,
        Delivery_Distance_km       REAL,
        Coupon_Used                TEXT
    );
""")

conn.commit()
print("✅ Banco e tabelas criados com sucesso.")

## 📥 INGESTÃO — Inserção das Dimensões

Extraímos os registros únicos de `CLIENTES`, `RESTAURANTES` e
`ENTREGADORES` a partir do DataFrame e inserimos no banco.

Como `City` e `Payment_Method` variam por pedido, consolidamos um valor
único por cliente usando a **moda** (valor mais frequente).

In [ ]:
clientes = (
    df.groupby("User_ID")
      .agg(
          City=("City", lambda x: x.mode()[0]),
          Payment_Method=("Payment_Method", lambda x: x.mode()[0])
      )
      .reset_index()
)
clientes.to_sql("CLIENTES", conn, if_exists="append", index=False)
print(f"✅ CLIENTES     → {len(clientes):>6,} registros inseridos")

# RESTAURANTES
restaurantes = (
    df[["Restaurant_ID", "City", "Restaurant_Lat", "Restaurant_Lon"]]
    .drop_duplicates(subset="Restaurant_ID")
)
restaurantes.to_sql("RESTAURANTES", conn, if_exists="append", index=False)
print(f"✅ RESTAURANTES → {len(restaurantes):>4,} registros inseridos")

# ENTREGADORES
entregadores = (
    df[["Driver_ID", "Driver_Vehicle", "Driver_Availability"]]
    .drop_duplicates(subset="Driver_ID")
)
entregadores.to_sql("ENTREGADORES", conn, if_exists="append", index=False)
print(f"✅ ENTREGADORES → {len(entregadores):>4,} registros inseridos")

## 📥 INGESTÃO — Inserção dos Pedidos (Tabela Fato)

Inserimos os **100.000 pedidos** na tabela fato, em lotes de 10.000 linhas
para evitar sobrecarga de memória.

In [ ]:
colunas_pedidos = [
    "Order_ID", "User_ID", "Restaurant_ID", "Driver_ID",
    "Item_Name", "Quantity", "Total_Price",
    "Order_Time", "Delivery_Duration_Minutes",
    "Order_Status", "Delivery_Distance_km", "Coupon_Used"
]

pedidos = df[colunas_pedidos].copy()

CHUNK = 10_000
for inicio in range(0, len(pedidos), CHUNK):
    lote = pedidos.iloc[inicio: inicio + CHUNK]
    lote.to_sql("PEDIDOS", conn, if_exists="append", index=False)

conn.commit()
print(f"✅ PEDIDOS → {len(pedidos):,} registros inseridos")

## ✅ Verificação — Comprovação dos 5.000+ Registros

Contagem de registros em cada tabela, confirmando que o banco foi
populado corretamente.

In [ ]:
print("=" * 45)
print("  📊  VERIFICAÇÃO DO BANCO — logisfast.db")
print("=" * 45)

for tabela in ["CLIENTES", "RESTAURANTES", "ENTREGADORES", "PEDIDOS"]:
    count = cursor.execute(f"SELECT COUNT(*) FROM {tabela}").fetchone()[0]
    marca = " ✅" if count >= 5000 else ""
    print(f"  {tabela:<15} : {count:>7,} registros{marca}")

## 🔗 EXTRAÇÃO (Regra de Ouro) — Query SQL com JOIN

Esta é a etapa exigida pela professora: os dados **saem do banco via
SQL**, nunca direto do CSV. A query abaixo faz `JOIN` entre `PEDIDOS` e
`CLIENTES`, filtra apenas pedidos entregues (`WHERE`) e agrega por cliente
(`GROUP BY`) calculando as métricas que alimentarão o K-Means:

- **frequencia** → quantos pedidos o cliente fez
- **ticket_medio** → valor médio gasto por pedido
- **tempo_entrega_medio** → tempo médio de entrega recebido
- **taxa_uso_cupom** → percentual de pedidos com cupom aplicado

In [ ]:
query_sql = """
    SELECT
        C.User_ID,
        C.City,
        C.Payment_Method,
        COUNT(P.Order_ID)                                  AS frequencia,
        ROUND(AVG(P.Total_Price), 2)                       AS ticket_medio,
        ROUND(AVG(P.Delivery_Duration_Minutes), 1)         AS tempo_entrega_medio,
        ROUND(100.0 * SUM(CASE WHEN P.Coupon_Used = 'Yes' THEN 1 ELSE 0 END)
              / COUNT(P.Order_ID), 1)                      AS taxa_uso_cupom
    FROM PEDIDOS P
    JOIN CLIENTES C ON P.User_ID = C.User_ID
    WHERE P.Order_Status = 'Delivered'
    GROUP BY C.User_ID, C.City, C.Payment_Method
    ORDER BY frequencia DESC
"""

df_extraido = pd.read_sql_query(query_sql, conn)

print(f"✅ Query executada via pd.read_sql_query()")
print(f"   Clientes extraídos: {len(df_extraido):,}\n")

display(df_extraido.head(10))

## 🧹 LIMPEZA — Tratamento Final no Pandas

Após extrair do banco, fazemos a limpeza final antes da matemática:

1. **Verificação de nulos** — clientes sem nenhum pedido entregue não
   aparecem na query (INNER JOIN já filtra isso), mas validamos de qualquer forma.
2. **Seleção das colunas numéricas** que entrarão no K-Means.
3. **Normalização (StandardScaler)** — o K-Means usa distância euclidiana,
   então todas as variáveis precisam estar na mesma escala. Sem isso,
   `ticket_medio` (valores até centenas) dominaria sobre `taxa_uso_cupom`
   (valores de 0 a 100) de forma desproporcional.

In [ ]:
print("🔍 Nulos no DataFrame extraído:")
print(df_extraido.isnull().sum())

# Seleção das features numéricas para o K-Means
features = ["frequencia", "ticket_medio", "tempo_entrega_medio", "taxa_uso_cupom"]
X = df_extraido[features].copy()

print(f"\n📐 Features selecionadas para o K-Means: {features}")
print(f"\nAntes da normalização:")
display(X.describe().round(2))

# Normalização
scaler = StandardScaler()
X_normalizado = scaler.fit_transform(X)

X_normalizado_df = pd.DataFrame(X_normalizado, columns=features)
print(f"\nApós StandardScaler (média ≈ 0, desvio padrão ≈ 1):")
display(X_normalizado_df.describe().round(2))

print("\n✅ Dados prontos para a Fase 4 — Modelagem com K-Means.")

conn.close()
print("🔒 Conexão com o banco encerrada.")

## Fase 4

## 📦 Importações Adicionais — Fase 4

Para a modelagem com K-Means, precisamos de bibliotecas adicionais:

- **sklearn.cluster.KMeans** → o algoritmo de agrupamento
- **matplotlib / seaborn** → visualização do Elbow Method e dos clusters

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Bibliotecas de modelagem importadas.")

## 📊 Elbow Method — Escolhendo o Número Ideal de Clusters (k)

O K-Means exige que definamos **antes** quantos grupos (k) queremos formar.
O Elbow Method testa vários valores de k e mede a **inércia** — soma das
distâncias entre cada ponto e o centro do seu cluster.

Quanto mais clusters, menor a inércia (cada ponto fica mais perto de "seu"
centro). Mas a partir de um certo k, a melhora passa a ser marginal —
esse ponto de inflexão (o "cotovelo" do gráfico) indica o k ideal.

Testamos de k=2 a k=8.

In [ ]:
inercias = []
valores_k = range(2, 9)

for k in valores_k:
    modelo_teste = KMeans(n_clusters=k, random_state=42, n_init=10)
    modelo_teste.fit(X_normalizado)
    inercias.append(modelo_teste.inertia_)
    print(f"  k={k}  →  inércia = {modelo_teste.inertia_:,.1f}")

# Plot do Elbow Method
plt.figure(figsize=(9, 5))
plt.plot(valores_k, inercias, marker='o', linewidth=2, markersize=8, color='#00b4d8')
plt.xlabel("Número de Clusters (k)")
plt.ylabel("Inércia")
plt.title("Elbow Method — Escolha do k Ideal")
plt.grid(alpha=0.3)
plt.xticks(list(valores_k))
plt.tight_layout()
plt.savefig("elbow_method.png", dpi=150)
plt.show()

print("\n💡 Observe o gráfico: o 'cotovelo' (ponto onde a curva para de cair")
print("   bruscamente) indica o k ideal. Neste projeto, escolhemos k=4.")

## 🌀 Treinamento do K-Means

Com base no Elbow Method, definimos **k=4** clusters.

**Justificativa de negócio para k=4:** a LogisFast precisa de poucos
grupos para que as campanhas de marketing sejam acionáveis na prática —
4 perfis distintos é um número operacionalmente viável para a equipe
de marketing criar 4 estratégias de cupom diferentes.

Parâmetros utilizados:
| Parâmetro | Valor | Significado |
|---|---|---|
| `n_clusters` | 4 | Número de grupos definido pelo Elbow Method |
| `random_state` | 42 | Garante reprodutibilidade dos resultados |
| `n_init` | 10 | Roda o algoritmo 10 vezes com centróides iniciais diferentes e mantém o melhor resultado |

In [ ]:
K_ESCOLHIDO = 4

modelo_kmeans = KMeans(
    n_clusters = K_ESCOLHIDO,
    random_state = 42,
    n_init = 10
)

modelo_kmeans.fit(X_normalizado)

print(f"✅ Modelo K-Means treinado com k={K_ESCOLHIDO}")
print(f"   Inércia final: {modelo_kmeans.inertia_:,.1f}")
print(f"   Iterações até convergir: {modelo_kmeans.n_iter_}")

## 🏷️ Atribuição dos Clusters aos Clientes

Usamos `.predict()` para descobrir a qual cluster cada cliente pertence,
e adicionamos essa informação de volta ao DataFrame original (não
normalizado) — para conseguirmos interpretar os clusters com os valores
reais (e não os valores normalizados, que não têm significado direto
para o negócio).

In [ ]:
df_extraido["Cluster"] = modelo_kmeans.labels_

print("✅ Cluster atribuído a cada cliente.\n")
print("📊 Distribuição de clientes por cluster:")
distrib = df_extraido["Cluster"].value_counts().sort_index()
for cluster, qtd in distrib.items():
    pct = qtd / len(df_extraido) * 100
    barra = "█" * int(pct / 2)
    print(f"  Cluster {cluster} : {qtd:>5,} clientes  ({pct:.1f}%)  {barra}")

display(df_extraido.head(10))

## Etapa 5

## 📦 Importações Adicionais — Fase 5

- **sklearn.metrics.silhouette_score** → mede a qualidade da separação dos clusters
- **sklearn.decomposition.PCA** → reduz as 4 dimensões para 2, permitindo visualizar os clusters em um gráfico

In [ ]:
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

print("✅ Bibliotecas de avaliação importadas.")

## 📐 Silhouette Score — Qualidade da Separação dos Clusters

O Silhouette Score mede, para cada ponto, o quão bem ele se encaixa no
próprio cluster comparado aos clusters vizinhos. O valor varia de **-1 a 1**:

| Faixa | Interpretação |
|---|---|
| 0.71 a 1.00 | Estrutura de clusters forte |
| 0.51 a 0.70 | Estrutura razoável |
| 0.26 a 0.50 | Estrutura fraca, mas pode ser útil |
| ≤ 0.25 | Sem estrutura significativa — clusters muito sobrepostos |

Diferente da Acurácia (Classificação), aqui não existe "gabarito" —
o Silhouette mede apenas a coesão geométrica dos grupos.

In [ ]:
sil_score = silhouette_score(X_normalizado, df_extraido["Cluster"])

print(f"📊 Silhouette Score: {sil_score:.4f}\n")

if sil_score > 0.5:
    avaliacao = "✅ Estrutura razoável a forte — clusters bem separados."
elif sil_score > 0.25:
    avaliacao = "⚠️ Estrutura fraca — clusters existem mas com alguma sobreposição."
else:
    avaliacao = "⚠️ Clusters com sobreposição significativa — separação geométrica fraca."

print(avaliacao)
print("\n💡 Importante: mesmo com Silhouette baixo, os clusters podem ainda")
print("   ser ÚTEIS para o negócio se os perfis médios forem distintos o")
print("   suficiente para gerar ações de marketing diferentes (ver próxima célula).")

## 📊 Visualização dos Clusters (PCA 2D)

Como temos 4 dimensões (frequência, ticket médio, tempo de entrega, taxa
de cupom), não é possível visualizar diretamente em um gráfico 2D.

Usamos o **PCA (Principal Component Analysis)** para reduzir essas 4
dimensões a apenas 2 componentes principais, preservando o máximo de
variância possível — permitindo visualizar a separação dos clusters.
"""

In [ ]:
pca = PCA(n_components=2, random_state=42)
componentes = pca.fit_transform(X_normalizado)

df_extraido["PCA1"] = componentes[:, 0]
df_extraido["PCA2"] = componentes[:, 1]

variancia_explicada = pca.explained_variance_ratio_.sum() * 100
print(f"📐 Variância explicada pelos 2 componentes: {variancia_explicada:.1f}%")

plt.figure(figsize=(9, 7))
cores = ['#00b4d8', '#f72585', '#7209b7', '#43aa8b']

for cluster in sorted(df_extraido["Cluster"].unique()):
    subset = df_extraido[df_extraido["Cluster"] == cluster]
    plt.scatter(subset["PCA1"], subset["PCA2"],
                label=f"Cluster {cluster}", alpha=0.5, s=20, color=cores[cluster])

plt.scatter(pca.transform(modelo_kmeans.cluster_centers_)[:, 0],
            pca.transform(modelo_kmeans.cluster_centers_)[:, 1],
            marker='X', s=300, color='black', label='Centróides', edgecolors='white', linewidths=2)

plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.title(f"Clusters de Clientes — LogisFast (k={K_ESCOLHIDO})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("clusters_visualizacao.png", dpi=150)
plt.show()

## 💡 Interpretação de Negócio — Perfil de Cada Cluster

Aqui traduzimos os números em **perfis de cliente acionáveis** para o
time de marketing. Calculamos a média de cada métrica por cluster e
damos um nome de negócio para cada grupo.

In [ ]:
features = ["frequencia", "ticket_medio", "tempo_entrega_medio", "taxa_uso_cupom"]
perfil_clusters = df_extraido.groupby("Cluster")[features].mean().round(2)
perfil_clusters["qtd_clientes"] = df_extraido["Cluster"].value_counts().sort_index()

print("📊 Perfil médio de cada cluster:\n")
display(perfil_clusters)

print("\n" + "=" * 70)
print("  💡  TRADUÇÃO PARA O NEGÓCIO")
print("=" * 70)

media_geral_cupom = df_extraido["taxa_uso_cupom"].mean()
media_geral_freq  = df_extraido["frequencia"].mean()

for cluster in perfil_clusters.index:
    linha = perfil_clusters.loc[cluster]
    qtd   = int(linha["qtd_clientes"])

    if linha["taxa_uso_cupom"] > media_geral_cupom and linha["frequencia"] < media_geral_freq:
        nome = "🎯 Caçador de Promoção"
        acao = "Reduzir cupons — esse cliente compra mesmo sem desconto agressivo."
    elif linha["frequencia"] > media_geral_freq and linha["taxa_uso_cupom"] < media_geral_cupom:
        nome = "💎 Cliente Fiel"
        acao = "Manter engajamento com programa de fidelidade, não com cupom."
    elif linha["ticket_medio"] > df_extraido["ticket_medio"].mean():
        nome = "👑 Alto Valor"
        acao = "Oferecer experiência premium, frete grátis em vez de desconto."
    else:
        nome = "😐 Cliente Casual"
        acao = "Campanhas de reativação com cupons pontuais."

    print(f"\n  Cluster {cluster} — {nome}  ({qtd:,} clientes)")
    print(f"    Frequência: {linha['frequencia']:.1f} pedidos | "
          f"Ticket: R$ {linha['ticket_medio']:.2f} | "
          f"Cupom: {linha['taxa_uso_cupom']:.1f}%")
    print(f"    ➜ Ação recomendada: {acao}")

print("\n" + "=" * 70)

## 💾 Exportação Final — Salvando os Resultados

Salvamos o DataFrame com os clusters atribuídos de volta no banco SQLite
(nova tabela `CLIENTES_SEGMENTADOS`) e baixamos o arquivo `.db` para
levar ao VS Code, onde o Dashboard em Streamlit vai consumi-lo.

In [ ]:
conn = sqlite3.connect(NOME_BANCO)

df_extraido.drop(columns=["PCA1", "PCA2"]).to_sql(
    "CLIENTES_SEGMENTADOS", conn, if_exists="replace", index=False
)
conn.commit()
conn.close()

print(f"✅ Tabela 'CLIENTES_SEGMENTADOS' salva em {NOME_BANCO}")
print(f"   {len(df_extraido):,} clientes com cluster atribuído.")

# Download do banco para uso no VS Code / Streamlit
# from google.colab import files
# files.download(NOME_BANCO)
print(f"\n📥 Download de '{NOME_BANCO}' iniciado — leve este arquivo para o VS Code.")